In [35]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [36]:
data = load_diabetes()
X, y = data['data'], data['target']
X.shape, y.shape

((442, 10), (442,))

In [37]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
# Standardize inputs and target
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test = y_scaler.transform(y_test.reshape(-1, 1))

In [39]:
# Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [40]:
# Define the MLP
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 5),
            nn.ReLU(),
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.network(x)

In [41]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [42]:
model = MLP(input_dim=10)

In [43]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [44]:
# Training loop
epochs = 50
model.train()

for epoch in range(epochs):

    y_hat = model(X_train)
    loss = criterion(y_hat, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs} | Train MSE: {loss.item():.2f}")

Epoch 1/50 | Train MSE: 1.04
Epoch 2/50 | Train MSE: 1.03
Epoch 3/50 | Train MSE: 1.03
Epoch 4/50 | Train MSE: 1.02
Epoch 5/50 | Train MSE: 1.01
Epoch 6/50 | Train MSE: 1.01
Epoch 7/50 | Train MSE: 1.00
Epoch 8/50 | Train MSE: 0.99
Epoch 9/50 | Train MSE: 0.99
Epoch 10/50 | Train MSE: 0.98
Epoch 11/50 | Train MSE: 0.98
Epoch 12/50 | Train MSE: 0.97
Epoch 13/50 | Train MSE: 0.96
Epoch 14/50 | Train MSE: 0.96
Epoch 15/50 | Train MSE: 0.95
Epoch 16/50 | Train MSE: 0.95
Epoch 17/50 | Train MSE: 0.94
Epoch 18/50 | Train MSE: 0.94
Epoch 19/50 | Train MSE: 0.93
Epoch 20/50 | Train MSE: 0.93
Epoch 21/50 | Train MSE: 0.92
Epoch 22/50 | Train MSE: 0.92
Epoch 23/50 | Train MSE: 0.91
Epoch 24/50 | Train MSE: 0.91
Epoch 25/50 | Train MSE: 0.90
Epoch 26/50 | Train MSE: 0.90
Epoch 27/50 | Train MSE: 0.89
Epoch 28/50 | Train MSE: 0.89
Epoch 29/50 | Train MSE: 0.88
Epoch 30/50 | Train MSE: 0.88
Epoch 31/50 | Train MSE: 0.88
Epoch 32/50 | Train MSE: 0.87
Epoch 33/50 | Train MSE: 0.87
Epoch 34/50 | Train

In [45]:
# Evaluation
model.eval()

# with torch.no_grad():
y_hat_test = model(X_test)
test_mse = criterion(y_hat_test, y_test)

y_hat_test = y_hat_test.detach().cpu().numpy().ravel()
y_test = y_test.cpu().numpy().ravel()
test_corr = np.corrcoef(y_hat_test, y_test)[0, 1]

print(f"Test MSE:  {test_mse.item():.2f}")
print(f"Test MSE:  {test_corr.item():.2f}")

Test MSE:  0.62
Test MSE:  0.55
